# RAG

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [4]:
# Create the document object
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
]

## Manual RAG

In [5]:
# Create the embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY,
)

In [6]:
# Create the vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['da25a2b4-3d2c-4afa-9ccd-07426019bb6b',
 'f7165422-315f-4c69-b2f5-1e664f887f62']

In [8]:
# Run the retriever

retriever = vector_store.as_retriever(
    search_kwargs = {"k":2}
)

In [9]:
# Invoke the retriever

query = "What is LangGraph"
retrieved_docs = retriever.invoke(query)

In [10]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    return "\n\n".join(
        doc.page_content
        for doc in documents
    )

"""
or
def docs_to_text(documents: list[Document])-> str:
    text = ""
    for doc in documents:
        text = text + doc.page_content + "\n\n"
    return text
"""

'\nor\ndef docs_to_text(documents: list[Document])-> str:\n    text = ""\n    for doc in documents:\n        text = text + doc.page_content + "\n\n"\n    return text\n'

In [11]:
context = format_docs(retrieved_docs)
print(context)

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [12]:
# Create the prompt

prompt = f"""
Answer the question using the following context.

Context:
{context}

Question:
{query}
"""

In [13]:
# Invoke the llm

response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'Ev4GCvsGARFNMg/eW+/JZBo0zxuj+4urP0JSgMdaICbECIw25g4UL4DYj7eDQ8E8TeaZRUyI+mgdsZOgV8x2FrbMvQhQiBt7TMKXgFCmKYiTdGuVsi94ijO5vM2pANeeJT+bcf83pZk8OryyeLEypl4dCQPO70MHCb4hIJFjN0SRpiQ/wBKLekaQ6emqtrBag6K7QhYULtslzdBbyl+2R7JrGROaERGhBZKLf5mFDtSJpmtj1Y+is71o7Tv0uiItm9JmsTPWOQjPPikLMD+ouIOJiAd71G5GlKj59rvUvzPtHfgPDKIvHiWF8nbU48HEN6zHU23EMU28zKa8qh45SxwrLZO6h013+hUUuAwLBBKsxcp+mQz1OMb0c2WtWLStS/Q2Ig5GQ+lphu20gwq0K5lD9UPNNgitqaO7rHjv83H3sUmr52vBH8fT7pEEN12R0N8ZVgn9Zouis1w/ZyJdgApgbhDQlYm9pn5tMIwMCpN6KtuoF/s/zDW9QQgVyOgEDxFO4xLqYyIuYwacTtCRFbwOkvwUwgmofPrQeRYkq3z+Ct3E2BOoI9A/G3LRWUGr0xT0mR8rvwwqWV1ibW/xoy/5evIHR5RHhmIcKpTooyJGspbScMBiwp8Fl6pqbFZXvXpcTro4PQw+TjZ2/uub2RwiYFfQ0YGxhkn35SMZcE8IPfnMkdebsqheDgQwLFqsAUpDtiEiiIQZv/iz1rmU/tt2n9wyLDvSSsbFivB3grEcpr3uwjoBn1O3mdc+0awXytMuHis1lIznQ1lK3KrSqPbagMJ+T85X1dkKHm+Pv6OFYCaMnC5PMquLm8gYefLRWIyctq9WKpVQO0tGpVB/gr1u05lqE3k

# RAG Chain with Runnables

In [14]:
query = "What is LangGraph?"

In [26]:
# Convert format_docs, prompt to runnable
from langchain_core.runnables import RunnableLambda

format_docs_runnable = RunnableLambda(format_docs)

In [24]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

retriever_chain = RunnableParallel(
    {
    "context": retriever | format_docs_runnable,
    "query": RunnablePassthrough()
    }
)

In [55]:
# Create prompt
def prompt_function (inputs):
    prompt = f"""
    Answer the question using the following context.

    Context:
    {inputs['context']}

    Question:
    {inputs['query']}
    """
    return prompt



In [56]:
# Debug 
retrieved = retriever_chain.invoke(query)
print(retrieved['context'])

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [57]:
# convert prompt_function to Runnable 
prompt_runnable = RunnableLambda(prompt_function)

In [58]:
# Create RAG Chain
RAG_Chain = retriever_chain | prompt_runnable | llm

In [59]:
response = RAG_Chain.invoke(query)

In [60]:
print(response.content)

[{'type': 'text', 'text': 'Based on the context provided, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'EpkGCpYGARFNMg+p2Sh1xjZNxxTU1Sf9NiZFFYwKpYK923kcAapvKpXqcuIrvK3aGyXUi3Ez488q7Yx1YL7dmzgjpWubrXPiNAQ0I97noSjGxu1M6wweu4sGr5IXyNqZL48yNNK3Kr2cXY1W163qz4A4rSjYIezCHd5SSCvc7sS17dl9/CxHgS6LDrGV9c6zCcjr+cSR8jAcdTNskcIQH78H3sZAzEmo/Bq3uM/8CFVmboJjGTPrkus/SaFI9rginno7ueZ0YieZBWFIVBxqtlKqlHubbr97T23mmiJMXssIzSDEjbORZsuvIAlDJcF0R2gB5oyiVyaYWu/47iUz0TtGawfGSV1F+VQgSDtl3TgXI6dzwQv2JPvu66E6vWGozXukwLa3vdp30vcAUiSvG5OV6n+NCwhQy73KUlxRwRvEgzmfIS2hJJKyugRbvgwAvgGCTm3xSHvHQ9siX7BBjCcecGQ25ReKdcYunBApj38xw/mvvFvpNMLFsP0Xy1O7bLCCozTVBKqbGSV3UXGJjGEItEeXo4k8fOtAP37MfJzKIC2AfC2ZyWpGbS8UYnpQUq9QFPDtxYi5xtwRPXAO95WMj02m6B5lc7hsru4Qgxd+1OeqbT+j9YZN4Y0lPNjZssBpbDNyvr8lFzYpbE92SgLBvSmoM4EzuRK4YPf+lQL973JcrY5d2t0wafov9jYLZf6mUVVfjArhStqdmr5y+DnbIKlen+UUuATj6XSldAjkzqpOM9+9uRwaxSYiFXuwdrU39sotTQoelINkbRrD3ju27jCCBdznOGsIG0m0UYHlQErj9Y2VqezpJWYS2boVsugo7/S5XZlfPwnALRGFR4WO2UEO6vO